In [38]:
import pandas as pd 
pd.set_option('display.max_columns', None)
import geopandas
import rasterio

In [39]:
def window_get_point_raster_values(raster, vector):
    vector_new = vector.copy().to_crs(raster.crs) 
    coord_list = [(x, y) for x, y in zip(vector_new["geometry"].x, vector_new["geometry"].y)]
    vector_new["value"] = [x for x in raster.sample(coord_list)]
    return vector_new


In [40]:
fires_with_factors = geopandas.read_file(r"C:\Users\jezkn\OneDrive\Documents\Birkbeck\Work\MSc Project\Wildfire Project\Outputs\VALIDATION_PartTwoOutputShape3.shp")
fires_with_factors.head()


,FIRE_ID,FIRE_TYPE,IG_DATE,UrbanAngle,Occurrence,point_orde,Y,IsUrban,WUIBreach,last_urban,geometry
0,CA3622612010420250902,Wildfire,2025-09-02,-163.647115,1,0,1,0,0,92.0,POINT (-2122095.131 1735488.284)
1,CA3622612010420250902,Wildfire,2025-09-02,-163.647115,1,1,1,0,0,92.0,POINT (-2122120.574 1735445.241)
2,CA3622612010420250902,Wildfire,2025-09-02,-163.647115,1,2,1,0,0,92.0,POINT (-2122146.018 1735402.199)
3,CA3622612010420250902,Wildfire,2025-09-02,-163.647115,1,3,1,0,0,92.0,POINT (-2122171.461 1735359.157)
4,CA3622612010420250902,Wildfire,2025-09-02,-163.647115,1,4,1,0,0,92.0,POINT (-2122196.904 1735316.115)


In [41]:
gee_df = geopandas.read_file(r"C:\Users\jezkn\OneDrive\Documents\Birkbeck\Work\MSc Project\Wildfire Project\Outputs\VALIDATION_PartThreeOutput2.csv")
gee_df.head()

,field_1,SR_B2,SR_B3,SR_B4,SR_B5,SR_B7,date_x,fire_id,occur_id,point_id,DEM,aspect,date_y,hillshade,slope,bi,date,erc,eto,fm100,fm1000,pr,rmax,rmin,th,tmmn,tmmx,vpd,vs,pop_density,u_class
0,0,8308,9251,9920,14829,12253,2025-01-29,VA3712108003520250129,9,0,531.721435546875,150,2025-01-29,150,20,56.0,2025-01-29,49.0,3.700000047683716,8.899999618530273,14.100000381469728,0.0,48.79999923706055,20.399999618530277,276.0,274.70001220703125,286.5,0.7400000095367432,7.800000190734863,12.811959266662598,11
1,1,8660,9633,10132,14535,11009,2025-01-29,VA3712108003520250129,9,1,522.3995971679688,231,2025-01-29,207,11,56.0,2025-01-29,49.0,3.700000047683716,8.899999618530273,14.100000381469728,0.0,48.79999923706055,20.399999618530277,276.0,274.70001220703125,286.5,0.7400000095367432,7.800000190734863,12.811959266662598,11
2,2,9166,10703,11823,16475,13181,2025-01-29,VA3712108003520250129,9,2,522.3995971679688,231,2025-01-29,207,11,56.0,2025-01-29,49.0,3.700000047683716,8.899999618530273,14.100000381469728,0.0,48.79999923706055,20.399999618530277,276.0,274.70001220703125,286.5,0.7400000095367432,7.800000190734863,12.811959266662598,11
3,3,9100,10519,11630,16871,13964,2025-01-29,VA3712108003520250129,9,3,530.7139282226562,226,2025-01-29,211,13,56.0,2025-01-29,49.0,3.700000047683716,8.899999618530273,14.100000381469728,0.0,48.79999923706055,20.399999618530277,276.0,274.70001220703125,286.5,0.7400000095367432,7.800000190734863,12.811959266662598,11
4,4,8067,9184,10233,14435,14275,2025-01-29,VA3712108003520250129,9,4,530.7139282226562,226,2025-01-29,211,13,56.0,2025-01-29,49.0,3.700000047683716,8.899999618530273,14.100000381469728,0.0,48.79999923706055,20.399999618530277,276.0,274.70001220703125,286.5,0.7400000095367432,7.800000190734863,12.811959266662598,11


In [42]:
gee_df.rename(columns={'fire_id': 'FIRE_ID'}, inplace=True)
fires_with_factors.rename(columns={'Occurrence': 'occur_id', 'point_orde': 'point_id'}, inplace=True)

In [43]:
gee_df = gee_df.astype({"FIRE_ID": "str", "occur_id": "str", "point_id": "str"})
fires_with_factors = fires_with_factors.astype({"FIRE_ID": "str", "occur_id": "str", "point_id": "str"})

In [44]:
fires_with_factors = pd.merge(fires_with_factors, gee_df, on=['FIRE_ID', 'occur_id', 'point_id'], how='left')

In [45]:
fires_with_factors['Month'] = pd.to_datetime(fires_with_factors['IG_DATE']).dt.month

In [46]:
lc = rasterio.open(r"C:\Users\jezkn\Local\Data Science Projects\MSc Project Data\Data\Land Cover\nlcd_2021_land_cover_test.tif")

In [47]:
fires_3 = window_get_point_raster_values(lc, fires_with_factors)

In [48]:
fires_3["LandCover"] = fires_3["value"].apply(lambda x: x[0])

In [49]:
def get_NDVI(row):
    if row['IG_DATE'] > '2013-05-17':
        NDVI = (row['SR_B5'] - row['SR_B4']) / (row['SR_B5'] + row['SR_B4'])
    elif (row['IG_DATE'] > '1999-07-28') & (row['IG_DATE'] < '2013-05-17'):
        NDVI = (row['SR_B4'] - row['SR_B3']) / (row['SR_B4'] + row['SR_B3'])
    elif row['IG_DATE'] < '1999-07-29':
        NDVI = (row['SR_B4'] - row['SR_B3']) / (row['SR_B4'] + row['SR_B3'])
    return NDVI

def get_NDWI(row):
    if row['IG_DATE'] > '2013-05-17':
        NDWI = (row['SR_B5'] - row['SR_B3']) / (row['SR_B5'] + row['SR_B3'])
    elif (row['IG_DATE'] > '1999-07-28') & (row['IG_DATE'] < '2013-05-17'):
        NDWI = (row['SR_B4'] - row['SR_B2']) / (row['SR_B4'] + row['SR_B2'])
    elif row['IG_DATE'] < '1999-07-29':
        NDWI = (row['SR_B4'] - row['SR_B2']) / (row['SR_B4'] + row['SR_B2'])
    return NDWI

def get_NBR(row):
    if row['IG_DATE'] > '2013-05-17':
        NBR = (row['SR_B5'] - row['SR_B7']) / (row['SR_B5'] + row['SR_B7'])
    elif (row['IG_DATE'] > '1999-07-28') & (row['IG_DATE'] < '2013-05-17'):
        NBR = (row['SR_B4'] - row['SR_B7']) / (row['SR_B4'] + row['SR_B7'])
    elif row['IG_DATE'] < '1999-07-29':
        NBR = (row['SR_B4'] - row['SR_B7']) / (row['SR_B4'] + row['SR_B7'])
    return NBR


In [50]:
fires_3['th'] = pd.to_numeric(fires_3["th"], errors="coerce")

In [51]:
fires_3['th'] = fires_3.apply(lambda x: x['th'] if x['th'] <= 180 else x['th']-360, axis=1)
fires_3.head()

,FIRE_ID,FIRE_TYPE,IG_DATE,UrbanAngle,occur_id,point_id,Y,IsUrban,WUIBreach,last_urban,geometry,field_1,SR_B2,SR_B3,SR_B4,SR_B5,SR_B7,date_x,DEM,aspect,date_y,hillshade,slope,bi,date,erc,eto,fm100,fm1000,pr,rmax,rmin,th,tmmn,tmmx,vpd,vs,pop_density,u_class,Month,value,LandCover
0,CA3622612010420250902,Wildfire,2025-09-02,-163.647115,1,0,1,0,0,92.0,POINT (-120.08429 36.23574),2093,9575,10385,11201,14885,14325,2025-09-02,102.35281372070312,56,2025-09-02,180,0,62.0,2025-09-02,84.0,7.400000095367432,6.5,6.900000095367432,0.0,53.59999847412109,12.100000381469728,-76.0,293.0,313.5,3.75,2.700000047683716,0.0,11,9,[81],81
1,CA3622612010420250902,Wildfire,2025-09-02,-163.647115,1,1,1,0,0,92.0,POINT (-120.08444 36.23531),2094,9702,10621,11636,14550,16347,2025-09-02,102.74132537841795,19,2025-09-02,180,0,62.0,2025-09-02,84.0,7.400000095367432,6.5,6.900000095367432,0.0,53.59999847412109,12.100000381469728,-76.0,293.0,313.5,3.75,2.700000047683716,0.0,11,9,[81],81
2,CA3622612010420250902,Wildfire,2025-09-02,-163.647115,1,2,1,0,0,92.0,POINT (-120.0846 36.23488),2095,9608,10671,11711,15241,15432,2025-09-02,102.74132537841795,19,2025-09-02,180,0,62.0,2025-09-02,84.0,7.400000095367432,6.5,6.900000095367432,0.0,53.59999847412109,12.100000381469728,-76.0,293.0,313.5,3.75,2.700000047683716,0.0,11,9,[82],82
3,CA3622612010420250902,Wildfire,2025-09-02,-163.647115,1,3,1,0,0,92.0,POINT (-120.08475 36.23445),2096,9659,10647,11687,15211,14978,2025-09-02,103.10577392578124,51,2025-09-02,179,0,62.0,2025-09-02,84.0,7.400000095367432,6.5,6.900000095367432,0.0,53.59999847412109,12.100000381469728,-76.0,293.0,313.5,3.75,2.700000047683716,0.0,11,9,[82],82
4,CA3622612010420250902,Wildfire,2025-09-02,-163.647115,1,4,1,0,0,92.0,POINT (-120.08491 36.23402),2097,9885,11084,12107,14797,15690,2025-09-02,103.28138732910156,341,2025-09-02,181,0,62.0,2025-09-02,84.0,7.400000095367432,6.5,6.900000095367432,0.0,53.59999847412109,12.100000381469728,-76.0,293.0,313.5,3.75,2.700000047683716,0.0,11,9,[82],82


In [52]:
fires_3.info(verbose=True, show_counts=True)

<class 'geopandas.geodataframe.GeoDataFrame'>
RangeIndex: 2387 entries, 0 to 2386
Data columns (total 42 columns):
 #   Column       Non-Null Count  Dtype   
---  ------       --------------  -----   
 0   FIRE_ID      2387 non-null   str     
 1   FIRE_TYPE    2387 non-null   str     
 2   IG_DATE      2387 non-null   str     
 3   UrbanAngle   2387 non-null   float64 
 4   occur_id     2387 non-null   str     
 5   point_id     2387 non-null   str     
 6   Y            2387 non-null   int64   
 7   IsUrban      2387 non-null   int64   
 8   WUIBreach    2387 non-null   int64   
 9   last_urban   2186 non-null   float64 
 10  geometry     2387 non-null   geometry
 11  field_1      2387 non-null   str     
 12  SR_B2        2387 non-null   str     
 13  SR_B3        2387 non-null   str     
 14  SR_B4        2387 non-null   str     
 15  SR_B5        2387 non-null   str     
 16  SR_B7        2387 non-null   str     
 17  date_x       2387 non-null   str     
 18  DEM          2387 no

In [53]:
fires_3 = fires_3.fillna(fires_3.groupby("occur_id").ffill())
fires_3 = fires_3.dropna(thresh=fires_3.shape[1])

In [54]:
fires_3.info(verbose=True, show_counts=True)

<class 'geopandas.geodataframe.GeoDataFrame'>
Index: 2186 entries, 0 to 2386
Data columns (total 42 columns):
 #   Column       Non-Null Count  Dtype   
---  ------       --------------  -----   
 0   FIRE_ID      2186 non-null   str     
 1   FIRE_TYPE    2186 non-null   str     
 2   IG_DATE      2186 non-null   str     
 3   UrbanAngle   2186 non-null   float64 
 4   occur_id     2186 non-null   str     
 5   point_id     2186 non-null   str     
 6   Y            2186 non-null   int64   
 7   IsUrban      2186 non-null   int64   
 8   WUIBreach    2186 non-null   int64   
 9   last_urban   2186 non-null   float64 
 10  geometry     2186 non-null   geometry
 11  field_1      2186 non-null   str     
 12  SR_B2        2186 non-null   str     
 13  SR_B3        2186 non-null   str     
 14  SR_B4        2186 non-null   str     
 15  SR_B5        2186 non-null   str     
 16  SR_B7        2186 non-null   str     
 17  date_x       2186 non-null   str     
 18  DEM          2186 non-nul

In [55]:
fires_3["SR_B2"] = pd.to_numeric(fires_3["SR_B2"], errors="coerce")
fires_3["SR_B3"] = pd.to_numeric(fires_3["SR_B3"], errors="coerce")
fires_3["SR_B4"] = pd.to_numeric(fires_3["SR_B4"], errors="coerce")
fires_3["SR_B5"] = pd.to_numeric(fires_3["SR_B5"], errors="coerce")
fires_3["SR_B7"] = pd.to_numeric(fires_3["SR_B7"], errors="coerce")
fires_3['aspect'] = pd.to_numeric(fires_3["aspect"], errors="coerce")
fires_3['hillshade'] = pd.to_numeric(fires_3["hillshade"], errors="coerce")
fires_3['slope'] = pd.to_numeric(fires_3["slope"], errors="coerce")
fires_3['bi'] = pd.to_numeric(fires_3["bi"], errors="coerce")
fires_3['erc'] = pd.to_numeric(fires_3["erc"], errors="coerce")
fires_3['eto'] = pd.to_numeric(fires_3["eto"], errors="coerce")
fires_3['fm100'] = pd.to_numeric(fires_3["fm100"], errors="coerce")
fires_3['fm1000'] = pd.to_numeric(fires_3["fm1000"], errors="coerce")
fires_3['pr'] = pd.to_numeric(fires_3["pr"], errors="coerce")
fires_3['rmax'] = pd.to_numeric(fires_3["rmax"], errors="coerce")
fires_3['rmin'] = pd.to_numeric(fires_3["rmin"], errors="coerce")
fires_3['tmmn'] = pd.to_numeric(fires_3["tmmn"], errors="coerce")
fires_3['tmmx'] = pd.to_numeric(fires_3["tmmx"], errors="coerce")
fires_3['vpd'] = pd.to_numeric(fires_3["vpd"], errors="coerce")
fires_3['vs'] = pd.to_numeric(fires_3["vs"], errors="coerce")
fires_3['pop_density'] = pd.to_numeric(fires_3["pop_density"], errors="coerce")
fires_3['u_class'] = pd.to_numeric(fires_3["u_class"], errors="coerce")


In [56]:
fires_3['NDVI'] = fires_3.apply(lambda x: get_NDVI(x), axis=1)
fires_3['NDWI'] = fires_3.apply(lambda x: get_NDWI(x), axis=1)
fires_3['NBR'] = fires_3.apply(lambda x: get_NBR(x), axis=1)

In [57]:
fires_3.head()

,FIRE_ID,FIRE_TYPE,IG_DATE,UrbanAngle,occur_id,point_id,Y,IsUrban,WUIBreach,last_urban,geometry,field_1,SR_B2,SR_B3,SR_B4,SR_B5,SR_B7,date_x,DEM,aspect,date_y,hillshade,slope,bi,date,erc,eto,fm100,fm1000,pr,rmax,rmin,th,tmmn,tmmx,vpd,vs,pop_density,u_class,Month,value,LandCover,NDVI,NDWI,NBR
0,CA3622612010420250902,Wildfire,2025-09-02,-163.647115,1,0,1,0,0,92.0,POINT (-120.08429 36.23574),2093,9575,10385,11201,14885,14325,2025-09-02,102.35281372070312,56,2025-09-02,180,0,62.0,2025-09-02,84.0,7.4,6.5,6.9,0.0,53.599998,12.1,-76.0,293.0,313.5,3.75,2.7,0.0,11,9,[81],81,0.141225,0.178077,0.019172
1,CA3622612010420250902,Wildfire,2025-09-02,-163.647115,1,1,1,0,0,92.0,POINT (-120.08444 36.23531),2094,9702,10621,11636,14550,16347,2025-09-02,102.74132537841795,19,2025-09-02,180,0,62.0,2025-09-02,84.0,7.4,6.5,6.9,0.0,53.599998,12.1,-76.0,293.0,313.5,3.75,2.7,0.0,11,9,[81],81,0.111281,0.156092,-0.058161
2,CA3622612010420250902,Wildfire,2025-09-02,-163.647115,1,2,1,0,0,92.0,POINT (-120.0846 36.23488),2095,9608,10671,11711,15241,15432,2025-09-02,102.74132537841795,19,2025-09-02,180,0,62.0,2025-09-02,84.0,7.4,6.5,6.9,0.0,53.599998,12.1,-76.0,293.0,313.5,3.75,2.7,0.0,11,9,[82],82,0.130974,0.176366,-0.006227
3,CA3622612010420250902,Wildfire,2025-09-02,-163.647115,1,3,1,0,0,92.0,POINT (-120.08475 36.23445),2096,9659,10647,11687,15211,14978,2025-09-02,103.10577392578124,51,2025-09-02,179,0,62.0,2025-09-02,84.0,7.4,6.5,6.9,0.0,53.599998,12.1,-76.0,293.0,313.5,3.75,2.7,0.0,11,9,[82],82,0.131013,0.176502,0.007718
4,CA3622612010420250902,Wildfire,2025-09-02,-163.647115,1,4,1,0,0,92.0,POINT (-120.08491 36.23402),2097,9885,11084,12107,14797,15690,2025-09-02,103.28138732910156,341,2025-09-02,181,0,62.0,2025-09-02,84.0,7.4,6.5,6.9,0.0,53.599998,12.1,-76.0,293.0,313.5,3.75,2.7,0.0,11,9,[82],82,0.099985,0.143464,-0.029291


fires_3.columns

In [58]:
print(len(fires_3))
#fires_3 = fires_3[fires_3["IG_DATE"] > '1994-12-31']

2186


In [59]:
fires_5 = fires_3[['FIRE_ID', 'UrbanAngle', 'occur_id', 'point_id', 'IsUrban', 'WUIBreach', 'Month', 'NDVI', 'NDWI', 'NBR', 'DEM', 'aspect', 'hillshade', 'slope', 'bi', 'erc', 'eto', 'fm100', 'fm1000', 'pr', 'rmax', 'rmin', 'th', 'pop_density', 'tmmn', 'tmmx', 'vpd', 'vs', 'LandCover', "u_class", 'Y', 'geometry']]

In [60]:
fires_5.head()

,FIRE_ID,UrbanAngle,occur_id,point_id,IsUrban,WUIBreach,Month,NDVI,NDWI,NBR,DEM,aspect,hillshade,slope,bi,erc,eto,fm100,fm1000,pr,rmax,rmin,th,pop_density,tmmn,tmmx,vpd,vs,LandCover,u_class,Y,geometry
0,CA3622612010420250902,-163.647115,1,0,0,0,9,0.141225,0.178077,0.019172,102.35281372070312,56,180,0,62.0,84.0,7.4,6.5,6.9,0.0,53.599998,12.1,-76.0,0.0,293.0,313.5,3.75,2.7,81,11,1,POINT (-120.08429 36.23574)
1,CA3622612010420250902,-163.647115,1,1,0,0,9,0.111281,0.156092,-0.058161,102.74132537841795,19,180,0,62.0,84.0,7.4,6.5,6.9,0.0,53.599998,12.1,-76.0,0.0,293.0,313.5,3.75,2.7,81,11,1,POINT (-120.08444 36.23531)
2,CA3622612010420250902,-163.647115,1,2,0,0,9,0.130974,0.176366,-0.006227,102.74132537841795,19,180,0,62.0,84.0,7.4,6.5,6.9,0.0,53.599998,12.1,-76.0,0.0,293.0,313.5,3.75,2.7,82,11,1,POINT (-120.0846 36.23488)
3,CA3622612010420250902,-163.647115,1,3,0,0,9,0.131013,0.176502,0.007718,103.10577392578124,51,179,0,62.0,84.0,7.4,6.5,6.9,0.0,53.599998,12.1,-76.0,0.0,293.0,313.5,3.75,2.7,82,11,1,POINT (-120.08475 36.23445)
4,CA3622612010420250902,-163.647115,1,4,0,0,9,0.099985,0.143464,-0.029291,103.28138732910156,341,181,0,62.0,84.0,7.4,6.5,6.9,0.0,53.599998,12.1,-76.0,0.0,293.0,313.5,3.75,2.7,82,11,1,POINT (-120.08491 36.23402)


In [61]:
fires_5 = fires_5.fillna(fires_5.groupby("occur_id").ffill())
fires_5["u_class"] = fires_5["u_class"].fillna(-200)

In [62]:
fires_5 = fires_5.dropna(thresh=fires_5.shape[1])

In [63]:
fires_5 = pd.get_dummies(fires_5, columns=['u_class'], dtype=int)

In [64]:
print(len(fires_5))

2186


In [65]:
first_urban_point = (fires_5[fires_5["IsUrban"] == 1].groupby("occur_id")["point_id"].min())

In [66]:
fires_5["first_urban_point"] = fires_5["occur_id"].map(first_urban_point)

In [67]:
fires_5["RemoveFlag"] = (fires_5["point_id"]).astype(int) > ((fires_5["first_urban_point"]).astype(int) + 5)

In [68]:
fires_5 = fires_5[fires_5["RemoveFlag"] == 0]

In [69]:
fires_ref = fires_5[['FIRE_ID', 'UrbanAngle', 'occur_id', 'point_id', 'IsUrban', 'WUIBreach', 'geometry', 'Y']]

In [70]:
fires_5.rename(columns={"u_class_10": "Water", "u_class_11": "Very_low_rural", "u_class_12": "Low_density_rural", "u_class_13": "Rural_cluster", "u_class_21": "Suburban", "u_class_22": "Semi_dense_urban", "u_class_23": "Dense_urban", "u_class_30": "Urban_centre", "u_class_-200.0": "No_Data_Urban_Class"}, inplace=True)
fires_5.drop(columns=['geometry'], inplace=True)

In [71]:
fires_5.insert(30, 'Water', 0)
fires_5.insert(37, 'Urban_centre', 0)
#fires_5["Urban_centre"] = 0

In [72]:
fires_5.info(verbose=True, show_counts=True)

<class 'geopandas.geodataframe.GeoDataFrame'>
Index: 1982 entries, 0 to 2386
Data columns (total 40 columns):
 #   Column             Non-Null Count  Dtype  
---  ------             --------------  -----  
 0   FIRE_ID            1982 non-null   str    
 1   UrbanAngle         1982 non-null   float64
 2   occur_id           1982 non-null   str    
 3   point_id           1982 non-null   str    
 4   IsUrban            1982 non-null   int64  
 5   WUIBreach          1982 non-null   int64  
 6   Month              1982 non-null   int32  
 7   NDVI               1982 non-null   float64
 8   NDWI               1982 non-null   float64
 9   NBR                1982 non-null   float64
 10  DEM                1982 non-null   str    
 11  aspect             1982 non-null   int64  
 12  hillshade          1982 non-null   int64  
 13  slope              1982 non-null   int64  
 14  bi                 1982 non-null   float64
 15  erc                1982 non-null   float64
 16  eto                19

In [98]:
fires_5.to_csv(r"C:\Users\jezkn\OneDrive\Documents\Birkbeck\Work\MSc Project\Wildfire Project\Outputs\VALIDATION_PartFourOutput_WUI.csv")

In [99]:
fires_ref.to_file(r"C:\Users\jezkn\OneDrive\Documents\Birkbeck\Work\MSc Project\Wildfire Project\Outputs\VALIDATION_PartFourOutput_ref_WUI.shp")